# DynaLoRA replication: training seeds and a larger base model (Colab, GPU)

Adds two training seeds (43, 44; the original adapters use 42) for the balanced IMDB and PubMed adapters on GPT-2 small, and the same two configurations on Qwen2.5-1.5B, with logit-space and weight-space generation, teacher-forced likelihood, zero-shot benchmarks, and re-scoring with SmolLM2-1.7B. Results go to the same Drive folder as the other notebooks, and every step skips what already exists, so the notebook can be re-run after a disconnection. Expected time on a T4: about 5 to 6 hours (faster on an L4 or A100).

In [ ]:
!nvidia-smi -L
!git clone -b paper https://github.com/alexroehrs/dynalora.git /content/dynalora
%cd /content/dynalora
!git pull -q
!pip install -q -r requirements.txt lm-eval
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT='/content/drive/MyDrive/dynalora_results'
import os; print(os.listdir(OUT))

## 1. GPT-2 small: two more training seeds

In [ ]:
!python scripts/run_all.py --out $OUT/seeds --models gpt2 --configs imdb_bal__seed43 imdb_bal__seed44 pubmed__seed43 pubmed__seed44 --methods dynalora weight --clf_configs imdb_bal yelp pubmed --n_samples 20
!python scripts/likelihood_tradeoff.py --model gpt2 --adapters_dir $OUT/seeds/adapters --configs imdb_bal__seed43 imdb_bal__seed44 pubmed__seed43 pubmed__seed44 --out $OUT/likelihood
!python scripts/zeroshot_bench.py --model gpt2 --adapters_dir $OUT/seeds/adapters --configs imdb_bal__seed43 imdb_bal__seed44 pubmed__seed43 pubmed__seed44 --out $OUT/bench

## 2. Qwen2.5-1.5B: balanced IMDB and PubMed

In [ ]:
!python scripts/run_all.py --out $OUT/qwen15 --models Qwen/Qwen2.5-1.5B --configs imdb_bal pubmed --methods dynalora weight --clf_configs imdb_bal yelp pubmed --n_samples 20

In [ ]:
!python scripts/likelihood_tradeoff.py --model Qwen/Qwen2.5-1.5B --adapters_dir $OUT/qwen15/adapters --configs imdb_bal pubmed --batch_size 8 --out $OUT/likelihood
!python scripts/zeroshot_bench.py --model Qwen/Qwen2.5-1.5B --adapters_dir $OUT/qwen15/adapters --configs imdb_bal pubmed --out $OUT/bench --batch_size 8 --limit 500 --lambdas 0 0.5 1 1.5 2 2.5 3
!python scripts/likelihood_summary.py $OUT/likelihood > /dev/null

## 3. Re-scoring with SmolLM2-1.7B and the unified domain classifier

In [ ]:
!python scripts/rescore.py --inputs "$OUT/*/generations_all_metrics.csv" --out $OUT/rescore

## 4. Package the new results

In [ ]:
!cd $OUT && zip -qr /content/dynalora_replication.zip seeds qwen15 rescore/rescore__seeds.csv.gz rescore/rescore__qwen15.csv.gz bench likelihood -x '*/adapters/*'
from google.colab import files
files.download('/content/dynalora_replication.zip')